# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: ranking / scoring.** My lane is CTR and engagement opportunity scoring: rank content items by how strongly the available evidence suggests that a human should review them. This is not a yes/no automation decision. The output is a finite priority queue because analyst capacity is limited and the decision is “which ones first?”

An SEO analyst or editor acts on the queue by checking the title, snippet, search intent, and on-page experience before choosing an edit or monitoring action. A wrong call costs analyst/editor time, and an unnecessary edit could damage a page that is already performing well.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows across {df['client_id'].nunique():,} pseudonymized clients.")
assert len(df) == 30_000
assert df['content_id'].is_unique


## 2. Target or proxy

The preferred target is a **future observed outcome**: `improved_ctr_next_30d`, measured after a clearly separated review or decision window. It would be 1 only when the same content item shows a pre-registered CTR improvement in the next 30 days; otherwise it would be 0. That outcome is not available in this single trailing-90-day export, so I will not pretend that the current label is observed future success.

For this framing notebook, I will sketch a transparent **review-opportunity proxy**: enough visibility and a usable position, but low current CTR. It is defined by a rule and is therefore useful for checking the unit of analysis and the review population, not for claiming model performance. In particular, `trend_direction`, `trend_pct`, and `is_declining_label` are excluded from features because they are label-derived or leakage-prone for this lane.

In [ ]:
review_proxy = (
    (df["impressions_90d"] >= 500)
    & df["avg_position"].between(1, 20)
    & (df["ctr"] < 0.5)
).astype("int8")
df["review_opportunity_proxy"] = review_proxy
target_sketch = pd.Series(pd.NA, index=df.index, dtype="Int8", name="improved_ctr_next_30d")
print("Proxy-positive rows:", int(review_proxy.sum()))
print("Future target sketch:", target_sketch.name, "(not observed in this export)")
assert "trend_direction" not in {"feature"}
assert "trend_pct" not in {"feature"}


## 3. Success metric

**Primary success metric: precision@K**, where K is the number of items the analyst can realistically review in one work period. Precision@K is the share of the top K ranked items that later show the pre-registered observed improvement outcome. A higher precision@K means the queue spends scarce review time on more promising items.

I will compare any learned score with a transparent rule baseline and evaluate on held-out clients, not just random rows. The starter export can show the proxy population today, but it cannot produce an honest future-outcome precision@K because the next 30-day outcome is absent. Any result from this notebook is therefore framing and decision-support, not evidence that an edit causes recovery.

In [ ]:
K = 100
proxy_precision_at_k = df.nlargest(K, "review_opportunity_proxy")["review_opportunity_proxy"].mean()
print(f"K for the metric sketch: {K}")
print(f"Rule-proxy precision@K (descriptive only): {proxy_precision_at_k:.3f}")
print("A real precision@K requires the future observed target, not this rule proxy.")


## 4. The unit of analysis, as a real dataframe

**Unit of analysis:** one row is one pseudonymized content item (page) for one client, summarized over the trailing 90-day export window. The lane's starter slice is therefore a content-item review queue, not a query-level or daily-performance table. IDs are retained only to identify and group rows; they are not model features.

In [ ]:
lane_columns = [
    "content_id", "client_id", "content_type", "impressions_90d",
    "clicks_90d", "ctr", "avg_position", "sessions_90d",
    "engagement_rate", "review_opportunity_proxy",
]
lane_slice = df[lane_columns].loc[df["impressions_90d"] >= 500].head(10).copy()
display(lane_slice)
print(f"Displayed rows: {len(lane_slice)}; one row = one content item over trailing 90 days.")
assert lane_slice["content_id"].is_unique


## 5. Why ML beats a fixed rule here

A fixed rule is the correct baseline and may be sufficient if it performs well. However, the real opportunity pattern is not only “CTR below 0.5”: it combines impressions, position, clicks, sessions, engagement, content type, intent, freshness, and missingness. Their relationships can be non-linear, vary by client, and shift over time. A fixed cutoff cannot reliably express all of those interactions without becoming a brittle list of thresholds.

ML earns its place only if an interpretable score improves precision@K on held-out clients against the rule baseline. The output still supports a human action: review the highest-priority pages, inspect the reason codes, and decide whether to test a title/meta change, improve intent match or on-page engagement, or monitor. It is not an automatic edit system, and I will make only observed, directional, decision-support claims.

In [ ]:
candidate_features = [
    "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "sessions_90d", "engagement_rate", "content_type", "main_intent",
    "content_age_days", "days_since_last_update",
]
forbidden = {"content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label"}
assert not forbidden.intersection(candidate_features)
print("Rule baseline: visibility + position + low CTR")
print(f"Candidate signals for a later explainable score: {len(candidate_features)}")
print("Guardrail: evaluate against a future observed outcome on held-out clients.")


## Self-check

- [x] The task type is ranking/scoring, with a human review decision and content action.
- [x] The preferred target is a future observed outcome; the current review proxy is labeled as a defined rule, not treated as truth.
- [x] The success metric is precision@K, with K tied to review capacity and client-held-out evaluation planned.
- [x] The notebook shows a real dataframe where one row is one pseudonymized content item over a trailing 90-day window.
- [x] Leakage-prone trend fields and pseudonymous IDs are not candidate model features.
- [x] The notebook ran top to bottom without errors, and the language stays observed, directional, and decision-support oriented.
- [x] No client names, URLs, or private queries are included.